In [1]:
from obspy import UTCDateTime
import obspy
from obspy.clients.fdsn import Client
import random
from obspy.clients.fdsn import RoutingClient
from obspy import Stream
from obspy.geodetics import gps2dist_azimuth
from obspy import signal
from obspy import read
import csv
import matplotlib.pyplot as plt
import folium
import numpy as np
import obspy
from scipy.fftpack import hilbert
#from obspy.signal.filter import envelope
from scipy.signal import resample

import json
import pandas as pd
from obspy.clients.fdsn.header import FDSNNoDataException
from scipy.signal import savgol_filter
import os
import folium
import numpy as np
import branca.colormap as cm
from scipy.signal import resample
from obspy import Stream
from matplotlib.colors import LogNorm

import pandas as pd
from shapely.geometry import box, LineString
from collections import defaultdict
from tqdm import tqdm
from processing_routines import * 
from plotting_routines import *
Dtmin_Noise=-25
Dtmax_Noise=-5
Dtmin_Pn=-5.
Dtmax_Pn=10.
Dtmin_Sn=-5.
Dtmax_Sn=10.

vLg_max=3.5
vLg_min=3.1
vLg=0.5*(vLg_max+vLg_min)
vPg_max=6.2
vPg_min=5.2
vPg=0.5*(vPg_max+vPg_min)
print(vPg)
#vPg=6.

tminCoda=300.
tmaxCoda=320.
import warnings
warnings.filterwarnings("ignore")

5.7


## Full Data processing until Tomography

In [4]:
from urllib.error import URLError
from http.client import HTTPException
import os
import warnings
warnings.filterwarnings("ignore")


def processing_data_routine(datacenters=['RESIF', 'ODC', 'ETH', 'INGV', 'GEOFON', 'IRIS', 'ICGC','LMU','BGR',"http://fdsnws.sismologia.ign.es"],
                            directory='/home/schreinl/Stage/Data1/',distmin=0.5,distmax=10.0,catalogue='/home/schreinl/Stage/Data/big_box_4.5.csv',
                            factor=1.1,fmin=[0.5,2,4,6],fmax=[1.5,3,6,8],codawindow='cutoff',snr_threshold=5):
    '''
    this function does all the heavy processing steps. It downloads first the data (all of it when the checking rhythm is not active),
    then subsequently the data is filtered, and the envelopes are calculated. then the SNR routine is started, removing data
    with a weak body wave SNR, and calculating the cutoff distance, on which the coda window is based. 
    
    '''
    
    
    eq_list = pd.read_csv(catalogue)
    #in order to run without checking the existence of the file, we have to write here 
    existent = False
    #and comment the checking rhythm
    for event in range(len(eq_list)):
        print("event", event, "out of", len(eq_list))
        start = UTCDateTime(eq_list["time"][event]) -25
        end = start + 700
        eq_lon = float(eq_list["longitude"][event])
        eq_lat = float(eq_list["latitude"][event])
        
        time_string = UTCDateTime.strftime(start, format="%Y_%m_%dT%H_%M_%S")
        #existent = True
        for i in range(len(fmin)):
            print(i)
            print(fmin[i])
            amplitude_test = f"/home/schreinl/Stage/Data1/{time_string}/{time_string}_new_{codawindow}_fac_{factor}_{fmin[i]}_{fmax[i]}Hz_dict.txt"
            #if os.path.exists(amplitude_test):
            #    continue
            #else:
                #existent = False
        
        retry_attempts = 3
        if existent == True:
            print(f'event {event} is already handled entirely')
            continue
        if existent == False:
            for attempt in range(retry_attempts):
                try:
                    st_all, stations_all, plot = big_downloader2(datacenters, start, end, eq_lon, eq_lat, distmin, distmax, directory, plot=False)
                    break
                except (URLError, FDSNNoDataException, HTTPException) as e:
                    print(f"Failed to download data for event {event} on attempt {attempt + 1}: {str(e)}")
                    if attempt < retry_attempts - 1:
                        print("Retrying with a smaller time window...")
                        end -= 100  # Reduce the time window by 100 seconds and retry
                    else:
                        print("Max retry attempts reached. Skipping event...")
                        continue




        for i in range(len(fmin)):
            print(f'Frequency range {fmin[i]}-{fmax[i]}')
            envelope_file = f"{directory}{time_string}/{time_string}_new_{codawindow}_fac_{factor}_{fmin[i]}_{fmax[i]}Hz_stream1.mseed"
            amplitude_file = f"{directory}{time_string}/{time_string}_new_{codawindow}_fac_{factor}_{fmin[i]}_{fmax[i]}Hz_dict1.txt"
            
            if os.path.exists(envelope_file) and os.path.exists(amplitude_file):
                print(f"Files for event {event} already exist. Skipping...")
                continue
            
            

            st_plot_filt_all = st_all.copy()
            st_plot_filt_all.filter("bandpass", freqmin=fmin[i], freqmax=fmax[i])

            st_envelope = obspy.Stream()
            smallest = 7000
            for tr in st_plot_filt_all:
                if tr.data is None or len(tr.data) == 0:
                    print(f"Skipping trace {tr.id} due to empty data.")
                    continue
                data_envelope = envelope_calculator(tr.data)
                npts = tr.stats.npts
                if npts >= smallest:
                    samprate = tr.stats.sampling_rate
                    t = np.arange(0, npts / samprate, 1 / samprate)
                    tr_envelope = obspy.Trace(data=data_envelope, header=tr.stats)
                    st_envelope.append(tr_envelope)

            snr_threshold = snr_threshold
            eq_start = start

            filtered_stations_with_SNR, stations_with_SNR, distance_dict, tcoda_test, filtered_st, stations_with_amps, amp_plot = SNR_all(
                stations_all, st_plot_filt_all, Dtmin_Pn, Dtmax_Pn, Dtmin_Sn, Dtmax_Sn, vLg_min, vLg_max, vPg_min, vPg_max, tminCoda, tmaxCoda,
                Dtmin_Noise, Dtmax_Noise, eq_start, eq_lat, eq_lon, snr_threshold=snr_threshold, plot_SNR=False, plot_amps=False, wavecode="Lg_Coda", dB=True, codawindow=codawindow, factor=factor)

            if filtered_stations_with_SNR is None or len(filtered_stations_with_SNR) == 0:
                continue

            with open(f"{directory}Dicts/{time_string}_{snr_threshold}_thresh_dict.txt", "w") as file:
                json.dump(distance_dict, file, indent=4)
            
            # Save stations_with_amps to a file
            with open(f"{directory}{time_string}/{time_string}_{fmin[i]}_{fmax[i]}Hz_{snr_threshold}_thresh_stations_with_amps1.txt", "w") as ampls:
                json.dump(stations_with_amps.tolist(), ampls, indent=4)

                # Save filtered stations with their corresponding SNR
            with open(f"{directory}{time_string}/{time_string}_{snr_threshold}_thresh_filtered_stations_SNR.txt", "w") as snrfile:
                json.dump(filtered_stations_with_SNR.tolist(), snrfile, indent=4)

                # Save the stations with SNR, unfiltered
            with open(f"{directory}{time_string}/{time_string}_unfiltered_stations_SNR.txt", "w") as unsnrfile:
                json.dump(stations_with_SNR.tolist(), unsnrfile, indent=4)
            amplitudes_full = calc_amps(stations_all, st_plot_filt_all, Dtmin_Pn, Dtmax_Pn, Dtmin_Sn, Dtmax_Sn, vLg_min,vLg_max,vPg_min,vPg_max, tcoda_test, tcoda_test+100, Dtmin_Noise, Dtmax_Noise, eq_start)
            amplitudes_small = calc_amps(stations_all, st_plot_filt_all, Dtmin_Pn, Dtmax_Pn, Dtmin_Sn, Dtmax_Sn, vLg_min,vLg_max,vPg_min,vPg_max, tcoda_test+80, tcoda_test+100, Dtmin_Noise, Dtmax_Noise, eq_start)
            SNR_dict = select_ratio_dict("Coda_Noise", amplitudes_full)
            SNR_dict_small = select_ratio_dict("Coda_Noise", amplitudes_small)
            envelopes_amps, st_smooth = envelopes_routine1(time_string, st_envelope, coda_dist_start=tcoda_test, coda_dist_end=tcoda_test + 100, plotting=False, method='cutoff', snr=SNR_dict, snr_window=SNR_dict_small)
            st_smooth.write(envelope_file, format="MSEED")

            with open(amplitude_file, "w") as ampls:
                json.dump(envelopes_amps, ampls, indent=4)

            filtered_station_names = set(row[1] for row in filtered_stations_with_SNR)
            filtered_smooth_stream = obspy.Stream()
            filtered_stream = obspy.Stream()
            for trace in st_smooth:
                if trace.stats.station in filtered_station_names:
                    filtered_smooth_stream.append(trace)
            for trace in st_envelope:
                if trace.stats.station in filtered_station_names:
                    filtered_stream.append(trace)
test = processing_data_routine(catalogue='/home/schreinl/Stage/Data/spain.csv')




event 0 out of 12
0
0.5
1
2
2
4
3
6
Earthquake at 2014-04-29T07:07:15.700000Z with magnitude 3.6


Processing stations of RESIF:   6%|▋         | 6/94 [00:03<00:56,  1.55it/s]

no Pn  0.553769965884501 list index out of range
no Sn  0.553769965884501 list index out of range


Processing stations of RESIF:  24%|██▍       | 23/94 [00:14<00:49,  1.43it/s]

no Sn  0.7709049708419617 list index out of range


...

no Sn  0.8795651609981086 list index out of range


Processing stations of INGV:   6%|▌         | 12/204 [00:11<03:48,  1.19s/it]

14 Trace(s) in Stream:
GU.IMI..BHZ | 2013-04-20T15:17:57.025000Z - 2013-04-20T15:20:28.825000Z | 20.0 Hz, 3037 samples
GU.IMI..BHZ | 2013-04-20T15:20:34.555000Z - 2013-04-20T15:22:32.405000Z | 20.0 Hz, 2358 samples
GU.IMI..BHZ | 2013-04-20T15:22:38.155000Z - 2013-04-20T15:23:21.855000Z | 20.0 Hz, 875 samples
GU.IMI..BHZ | 2013-04-20T15:23:27.595000Z - 2013-04-20T15:23:50.695000Z | 20.0 Hz, 463 samples
GU.IMI..BHZ | 2013-04-20T15:23:56.435000Z - 2013-04-20T15:25:50.135000Z | 20.0 Hz, 2275 samples
GU.IMI..BHZ | 2013-04-20T15:25:58.425000Z - 2013-04-20T15:25:58.425000Z | 20.0 Hz, 1 samples
GU.IMI..BHZ | 2013-04-20T15:26:00.035000Z - 2013-04-20T15:26:06.635000Z | 20.0 Hz, 133 samples
GU.IMI..BHZ | 2013-04-20T15:26:12.395000Z - 2013-04-20T15:26:51.995000Z | 20.0 Hz, 793 samples
GU.IMI..BHZ | 2013-04-20T15:26:57.715000Z - 2013-04-20T15:27:16.715000Z | 20.0 Hz, 381 samples
GU.IMI..BHZ | 2013-04-20T15:27:22.435000Z - 2013-04-20T15:27:37.035000Z | 20.0 Hz, 293 samples
GU.IMI..BHZ | 2013-04-20T1

Processing stations of LMU: 100%|██████████| 1/1 [00:00<00:00, 1778.75it/s]


KeyboardInterrupt: 

In [5]:
test = processing_data_routine(catalogue='/home/schreinl/Stage/Data/trial.csv')

event 0 out of 2
0
0.5
1
2
2
4
3
6
Earthquake at 2025-01-12T13:51:40.660000Z with magnitude 3.9


Processing stations of ODC:  69%|██████▉   | 116/167 [00:05<00:02, 18.30it/s]

no Sn  0.7855167640428073 list index out of range


Processing stations of ODC:  71%|███████   | 118/167 [00:05<00:02, 16.38it/s]

no Pn  0.6340014678311483 list index out of range
no Sn  0.6340014678311483 list index out of range


...

no Pn  0.7113915016414014 list index out of range
no Sn  0.7113915016414014 list index out of range


Processing stations of IRIS:  72%|███████▏  | 104/144 [00:04<00:01, 39.72it/s]

no Pn  0.7213748829921666 list index out of range
no Sn  0.7213748829921666 list index out of range
no Sn  0.7678401763245136 list index out of range


Processing stations of BGR: 100%|██████████| 196/196 [00:13<00:00, 14.62it/s]
Processing stations of http://fdsnws.sismologia.ign.es: 100%|██████████| 16/16 [00:01<00:00, 14.39it/s]

Frequency range 0.5-1.5
Files for event 1 already exist. Skipping...
Frequency range 2-3
Files for event 1 already exist. Skipping...
Frequency range 4-6
Files for event 1 already exist. Skipping...
Frequency range 6-8
Files for event 1 already exist. Skipping...
